# Data Exam (Medium) — CoderPad Practice Problems

**Format:** Two tiers of practice, matching real CoderPad dynamics.

**Part A: Warmups (8 problems, 5-10 min each)**
- Fundamentals that must be automatic — if these take > 10 min, drill them first
- Softmax, cosine similarity, one-hot, running stats, top-k, cross-entropy, layer norm, confusion matrix

**Part B: Full Problems (8 problems, 30 min each)**
- Simulate actual CoderPad sessions — set a timer, no docs, no autocomplete
- Every problem gives you the full spec (formulas, interface, edge cases) — just like the real CoderPad
- The challenge is clean execution under pressure, not memorization
- Sorted by likelihood for a datasets-focused MTS first round

**Rules:**
- Set a timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass
- If you finish early, review edge cases

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Any
from collections import Counter
import math

---

# Part A: Warmups (5-10 min each)

These must be automatic. If any take > 10 min, that's priority #1 to drill.

---

## Warmup 1: Stable Softmax

Implement softmax that doesn't overflow on large inputs.

In [ ]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Numerically stable softmax. No torch.softmax or F.softmax."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 1 ---
x = torch.tensor([1.0, 2.0, 3.0])
s = stable_softmax(x)
assert torch.allclose(s, F.softmax(x, dim=-1), atol=1e-5)
assert abs(s.sum().item() - 1.0) < 1e-5, "Should sum to 1"

# Stability test — this would overflow with naive exp
big = torch.tensor([1000.0, 1001.0, 1002.0])
s_big = stable_softmax(big)
assert not torch.isnan(s_big).any(), "Should not produce NaN on large inputs"
assert not torch.isinf(s_big).any(), "Should not produce Inf on large inputs"
assert abs(s_big.sum().item() - 1.0) < 1e-5

# Batch
batch = torch.randn(4, 10)
s_batch = stable_softmax(batch, dim=-1)
assert s_batch.shape == (4, 10)
assert torch.allclose(s_batch.sum(dim=-1), torch.ones(4), atol=1e-5)

print("Warmup 1: PASSED")

---

## Warmup 2: Cosine Similarity Matrix

Given two batches of vectors, compute the full pairwise cosine similarity matrix.

In [ ]:
def cosine_similarity_matrix(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Pairwise cosine similarity. a: (N, D), b: (M, D) -> (N, M). Handle zero vectors."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 2 ---
a = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
b = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
sim = cosine_similarity_matrix(a, b)
assert sim.shape == (3, 2)
assert abs(sim[0, 0].item() - 1.0) < 1e-5  # identical
assert abs(sim[0, 1].item() - 0.0) < 1e-5  # orthogonal
assert abs(sim[2, 0].item() - 1/math.sqrt(2)) < 1e-5

# Zero vector
c = torch.tensor([[0.0, 0.0], [1.0, 0.0]])
sim_z = cosine_similarity_matrix(c, b)
assert sim_z[0, 0].item() == 0.0, "Zero vector sim should be 0"
assert sim_z[0, 1].item() == 0.0

# Larger batch
a_big = torch.randn(100, 64)
b_big = torch.randn(50, 64)
sim_big = cosine_similarity_matrix(a_big, b_big)
assert sim_big.shape == (100, 50)
assert sim_big.min() >= -1.0 - 1e-5
assert sim_big.max() <= 1.0 + 1e-5

print("Warmup 2: PASSED")

---

## Warmup 3: One-Hot Encoding

Convert integer class labels to one-hot vectors. No `F.one_hot`.

In [ ]:
def one_hot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    """labels: (B,) int64 -> (B, num_classes) float32 one-hot. No F.one_hot."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 3 ---
labels = torch.tensor([0, 2, 1, 4])
oh = one_hot(labels, num_classes=5)
assert oh.shape == (4, 5)
assert oh.dtype == torch.float32
assert oh[0].tolist() == [1, 0, 0, 0, 0]
assert oh[1].tolist() == [0, 0, 1, 0, 0]
assert oh[3].tolist() == [0, 0, 0, 0, 1]
assert (oh.sum(dim=-1) == 1).all(), "Each row should sum to 1"

# Single element
oh_single = one_hot(torch.tensor([0]), 3)
assert oh_single.shape == (1, 3)

print("Warmup 3: PASSED")

---

## Warmup 4: Running Statistics Tracker

Track mean and variance online (one sample at a time). Useful for dataset normalization.

In [ ]:
class RunningStats:
    """Track mean and variance incrementally (Welford's algorithm).
    
    - update(x): add a single value (float or tensor)
    - mean -> current mean
    - variance -> current population variance
    - std -> current population std
    - count -> number of samples seen
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 4 ---
rs = RunningStats()
data = [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]
for x in data:
    rs.update(x)

assert rs.count == 8
assert abs(rs.mean - 5.0) < 1e-5, f"Mean should be 5.0, got {rs.mean}"
assert abs(rs.variance - 4.0) < 1e-5, f"Variance should be 4.0, got {rs.variance}"
assert abs(rs.std - 2.0) < 1e-5, f"Std should be 2.0, got {rs.std}"

# Verify against numpy
data_t = torch.tensor(data)
assert abs(rs.mean - data_t.mean().item()) < 1e-5
assert abs(rs.variance - data_t.var(correction=0).item()) < 1e-5

# Works with tensors too
rs2 = RunningStats()
for x in torch.randn(100):
    rs2.update(x.item())
assert rs2.count == 100
assert abs(rs2.mean) < 0.5  # roughly centered near 0

print("Warmup 4: PASSED")

---

## Warmup 5: Top-K with Indices

Return the top-k values and their original indices from an unsorted tensor. No `torch.topk`.

In [ ]:
def top_k(x: torch.Tensor, k: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (values, indices) of the k largest elements. x is 1D. No torch.topk."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 5 ---
x = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0])
vals, idxs = top_k(x, 3)
assert vals.tolist() == [9.0, 6.0, 5.0], f"Expected [9,6,5], got {vals.tolist()}"
assert idxs.tolist() == [5, 7, 4], f"Expected [5,7,4], got {idxs.tolist()}"

# k = 1
vals1, idxs1 = top_k(x, 1)
assert vals1.item() == 9.0
assert idxs1.item() == 5

# k = len(x)
vals_all, idxs_all = top_k(x, len(x))
assert len(vals_all) == len(x)

# Verify values match indices
assert torch.allclose(x[idxs], vals)

print("Warmup 5: PASSED")

---

## Warmup 6: Cross-Entropy Loss

Implement cross-entropy loss from scratch given logits and integer labels. No `F.cross_entropy`.

In [ ]:
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """logits: (B, C), targets: (B,) int64. Return scalar mean loss. No F.cross_entropy.
    Must be numerically stable (use log-sum-exp trick)."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 6 ---
logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
targets = torch.tensor([0, 1])
loss = cross_entropy(logits, targets)
expected = F.cross_entropy(logits, targets)
assert abs(loss.item() - expected.item()) < 1e-5, f"Expected {expected.item()}, got {loss.item()}"

# Numerical stability
big_logits = torch.tensor([[1000.0, 999.0], [0.0, 1000.0]])
big_targets = torch.tensor([0, 1])
big_loss = cross_entropy(big_logits, big_targets)
assert not torch.isnan(big_loss), "Should handle large logits"
assert not torch.isinf(big_loss), "Should handle large logits"

# Gradient flows
logits_g = torch.randn(4, 5, requires_grad=True)
targets_g = torch.randint(0, 5, (4,))
loss_g = cross_entropy(logits_g, targets_g)
loss_g.backward()
assert logits_g.grad is not None

print("Warmup 6: PASSED")

---

## Warmup 7: Layer Normalization

Implement LayerNorm from scratch. No `nn.LayerNorm`.

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm over the last dimension. Learnable gamma (weight) and beta (bias).
    No nn.LayerNorm or F.layer_norm."""
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Warmup 7 ---
torch.manual_seed(42)
ln = LayerNorm(64)
ref_ln = nn.LayerNorm(64)

# Copy params for comparison
with torch.no_grad():
    ref_ln.weight.copy_(ln.weight)
    ref_ln.bias.copy_(ln.bias)

x = torch.randn(4, 10, 64)
out = ln(x)
ref_out = ref_ln(x)
assert out.shape == (4, 10, 64)
assert torch.allclose(out, ref_out, atol=1e-5), "Should match nn.LayerNorm output"

# Output should be normalized along last dim (before gamma/beta)
# With default gamma=1, beta=0, mean~=0 and var~=1
assert abs(out.mean(dim=-1).mean().item()) < 0.01
assert abs(out.var(dim=-1, correction=0).mean().item() - 1.0) < 0.05

# Gradient flows
loss = out.sum()
loss.backward()
assert ln.weight.grad is not None
assert ln.bias.grad is not None

print("Warmup 7: PASSED")

---

## Warmup 8: Confusion Matrix

Build a confusion matrix from predictions and labels. No sklearn.

In [ ]:
def confusion_matrix(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int) -> torch.Tensor:
    """Return (num_classes, num_classes) matrix where cm[true][pred] = count. No sklearn."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 8 ---
preds = torch.tensor([0, 1, 2, 0, 1, 2, 0, 0])
targs = torch.tensor([0, 1, 2, 1, 0, 2, 0, 2])
cm = confusion_matrix(preds, targs, num_classes=3)
assert cm.shape == (3, 3)
assert cm.sum().item() == 8, "Total should equal number of samples"

# Diagonal = correct predictions
assert cm[0, 0].item() == 2  # true=0, pred=0
assert cm[1, 1].item() == 1  # true=1, pred=1
assert cm[2, 2].item() == 2  # true=2, pred=2

# Off-diagonal = errors
assert cm[1, 0].item() == 1  # true=1, pred=0
assert cm[0, 1].item() == 1  # true=0, pred=1
assert cm[2, 0].item() == 1  # true=2, pred=0

# Row sums = number of true instances per class
assert cm[0].sum().item() == 3  # 3 true class-0 samples
assert cm[1].sum().item() == 2  # 2 true class-1 samples
assert cm[2].sum().item() == 3  # 3 true class-2 samples

print("Warmup 8: PASSED")

---

# Part B: Full Problems (30 min each)

Each problem gives you the complete spec — formulas, interface, edge cases — just like a real CoderPad prompt. The test is whether you can translate a clear spec into clean, working code under time pressure.

Set a 30-minute timer. No docs, no autocomplete.

---

## Problem 1: Video Clip Processor

**Likelihood: Very High** — The role is data curation for video models. This is the exact kind of thing you'd build day one.

### Scenario

You have a list of video clip metadata dicts. Build functions to filter, validate, and batch them.

### Requirements

**`validate_clips(clips: list[dict]) -> tuple[list[dict], list[dict]]`:**
- Each clip dict has keys: `"id"` (str), `"duration"` (float, seconds), `"resolution"` (tuple[int,int] as (H,W)), `"fps"` (float), `"caption"` (str), `"quality_score"` (float 0-1)
- A clip is **valid** if ALL of:
  - duration is between 1.0 and 30.0 seconds (inclusive)
  - resolution H and W are both >= 256
  - fps >= 15.0
  - caption is non-empty (after stripping whitespace)
  - quality_score >= 0.5
- Return `(valid_clips, rejected_clips)`

**`batch_clips(clips: list[dict], max_batch_duration: float) -> list[list[dict]]`:**
- Greedily pack clips into batches where total duration per batch <= max_batch_duration
- Process clips in the order given — don't reorder
- A clip that exceeds max_batch_duration by itself goes in its own batch
- Return list of batches (each batch is a list of clip dicts)

**`compute_batch_stats(batches: list[list[dict]]) -> list[dict]`:**
- For each batch, return a dict with:
  - `"num_clips"`: int
  - `"total_duration"`: float
  - `"avg_quality"`: float (mean quality_score)
  - `"resolutions"`: set of unique (H,W) tuples in the batch

In [ ]:
def validate_clips(clips: list[dict]) -> tuple[list[dict], list[dict]]:
    # YOUR CODE HERE
    pass


def batch_clips(clips: list[dict], max_batch_duration: float) -> list[list[dict]]:
    # YOUR CODE HERE
    pass


def compute_batch_stats(batches: list[list[dict]]) -> list[dict]:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
clips = [
    {"id": "a", "duration": 5.0, "resolution": (720, 1280), "fps": 30.0, "caption": "a red car driving", "quality_score": 0.9},
    {"id": "b", "duration": 0.5, "resolution": (720, 1280), "fps": 30.0, "caption": "too short", "quality_score": 0.8},  # duration < 1
    {"id": "c", "duration": 3.0, "resolution": (128, 128), "fps": 30.0, "caption": "too small", "quality_score": 0.7},  # resolution < 256
    {"id": "d", "duration": 10.0, "resolution": (512, 512), "fps": 24.0, "caption": "a sunset", "quality_score": 0.85},
    {"id": "e", "duration": 8.0, "resolution": (1080, 1920), "fps": 60.0, "caption": "  ", "quality_score": 0.95},  # empty caption
    {"id": "f", "duration": 4.0, "resolution": (480, 640), "fps": 30.0, "caption": "dog playing", "quality_score": 0.3},  # low quality
    {"id": "g", "duration": 7.0, "resolution": (720, 1280), "fps": 30.0, "caption": "ocean waves", "quality_score": 0.7},
    {"id": "h", "duration": 2.0, "resolution": (256, 256), "fps": 15.0, "caption": "minimal valid", "quality_score": 0.5},
]

valid, rejected = validate_clips(clips)
valid_ids = {c["id"] for c in valid}
rejected_ids = {c["id"] for c in rejected}
assert valid_ids == {"a", "d", "g", "h"}, f"Expected valid={{a,d,g,h}}, got {valid_ids}"
assert rejected_ids == {"b", "c", "e", "f"}, f"Expected rejected={{b,c,e,f}}, got {rejected_ids}"

# Batching
batches = batch_clips(valid, max_batch_duration=15.0)
# a(5) + d(10) = 15 fits; g(7) + h(2) = 9 fits
assert len(batches) == 2, f"Expected 2 batches, got {len(batches)}"
batch_durations = [sum(c["duration"] for c in b) for b in batches]
assert all(d <= 15.0 for d in batch_durations), f"Batch durations exceed max: {batch_durations}"

# Oversized single clip
big_clip = [{"id": "x", "duration": 20.0, "resolution": (720, 1280), "fps": 30.0, "caption": "long", "quality_score": 0.9}]
batches_big = batch_clips(big_clip, max_batch_duration=10.0)
assert len(batches_big) == 1, "Oversized clip should go in its own batch"
assert len(batches_big[0]) == 1

# Empty input
assert batch_clips([], max_batch_duration=10.0) == []

# Stats
stats = compute_batch_stats(batches)
assert len(stats) == 2
assert stats[0]["num_clips"] == 2
assert abs(stats[0]["total_duration"] - 15.0) < 1e-5
assert isinstance(stats[0]["resolutions"], set)
assert isinstance(stats[0]["avg_quality"], float)

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Embedding Retrieval with Metadata Filters

**Likelihood: Very High** — Runway's feature lakehouse is embedding-centric. Search + filter is the core query pattern.

### Scenario

You're building a simple search system over a dataset of items, each with an embedding vector and metadata tags.

### Requirements

**`SearchIndex`:**
- `__init__(self, dim: int)` — embedding dimension
- `add(self, items: list[dict])` — each dict has `"id"` (str), `"embedding"` (list[float] or tensor of length dim), `"tags"` (set[str]). Can be called multiple times.
- `search(self, query_embedding, k: int, require_tags: set[str] | None = None, exclude_tags: set[str] | None = None) -> list[dict]`:
  - Find top-k items by **cosine similarity** to query_embedding
  - If `require_tags` is provided, only include items that have ALL of those tags
  - If `exclude_tags` is provided, exclude items that have ANY of those tags
  - Return list of `{"id": str, "score": float}` dicts, sorted by descending score
  - **Cosine similarity formula:** `cos(a, b) = (a · b) / (||a|| * ||b||)`. If either vector is zero, similarity = 0.
- `__len__` — total items in index

**Important:**
- Use matrix operations for the similarity computation (not loops over individual items)
- Apply tag filters BEFORE computing similarities (don't compute similarity for filtered-out items)
- If k > number of matching items after filtering, return all matching items

In [ ]:
class SearchIndex:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 2 ---
idx = SearchIndex(dim=4)
assert len(idx) == 0

idx.add([
    {"id": "cat1", "embedding": [1.0, 0.0, 0.0, 0.0], "tags": {"animal", "indoor"}},
    {"id": "cat2", "embedding": [0.9, 0.1, 0.0, 0.0], "tags": {"animal", "outdoor"}},
    {"id": "car1", "embedding": [0.0, 1.0, 0.0, 0.0], "tags": {"vehicle", "outdoor"}},
    {"id": "car2", "embedding": [0.0, 0.9, 0.1, 0.0], "tags": {"vehicle", "indoor"}},
    {"id": "tree", "embedding": [0.0, 0.0, 1.0, 0.0], "tags": {"nature", "outdoor"}},
])
assert len(idx) == 5

# Basic search — query close to [1,0,0,0]
results = idx.search([1.0, 0.0, 0.0, 0.0], k=2)
assert len(results) == 2
assert results[0]["id"] == "cat1"
assert abs(results[0]["score"] - 1.0) < 1e-5
assert results[1]["id"] == "cat2"  # most similar after cat1

# Search with require_tags
results_outdoor = idx.search([1.0, 0.0, 0.0, 0.0], k=3, require_tags={"outdoor"})
result_ids = [r["id"] for r in results_outdoor]
assert "cat1" not in result_ids, "cat1 is indoor, should be filtered out"
assert "cat2" in result_ids

# Search with exclude_tags
results_no_vehicle = idx.search([0.0, 1.0, 0.0, 0.0], k=5, exclude_tags={"vehicle"})
result_ids_nv = [r["id"] for r in results_no_vehicle]
assert "car1" not in result_ids_nv
assert "car2" not in result_ids_nv

# Both filters together
results_both = idx.search([1.0, 0.0, 0.0, 0.0], k=5, require_tags={"outdoor"}, exclude_tags={"vehicle"})
result_ids_both = {r["id"] for r in results_both}
assert "car1" not in result_ids_both  # excluded by vehicle
assert "cat1" not in result_ids_both  # excluded by not outdoor
assert "cat2" in result_ids_both  # outdoor + animal

# k > matching items
results_big_k = idx.search([1.0, 0.0, 0.0, 0.0], k=100)
assert len(results_big_k) == 5

# Incremental add
idx.add([{"id": "dog", "embedding": [0.8, 0.2, 0.0, 0.0], "tags": {"animal", "outdoor"}}])
assert len(idx) == 6

# Results are sorted descending by score
results_sorted = idx.search([1.0, 0.0, 0.0, 0.0], k=6)
scores = [r["score"] for r in results_sorted]
assert scores == sorted(scores, reverse=True), "Results should be sorted by descending score"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Data Quality Report

**Likelihood: High** — Job description: "filtering and quality assurance." This is bread-and-butter data curation.

### Scenario

You receive a batch of training records. Before adding them to the training set, you need to generate a quality report identifying issues.

### Requirements

**`audit_dataset(records: list[dict], schema: dict) -> dict`:**

The `schema` dict defines expected fields and their constraints:
```python
schema = {
    "id": {"type": str, "required": True},
    "caption": {"type": str, "required": True, "min_length": 5},
    "width": {"type": int, "required": True, "min": 256, "max": 4096},
    "height": {"type": int, "required": True, "min": 256, "max": 4096},
    "quality_score": {"type": float, "required": True, "min": 0.0, "max": 1.0},
    "source": {"type": str, "required": False},
}
```

Return a report dict with:
- `"total_records"`: int
- `"valid_records"`: int (records with zero issues)
- `"issues"`: list of `{"record_index": int, "field": str, "issue": str}` dicts
  - Issue types: `"missing"` (required field absent or None), `"wrong_type"` (wrong Python type), `"out_of_range"` (below min or above max), `"too_short"` (string shorter than min_length)
- `"duplicate_ids"`: list of ID values that appear more than once
- `"field_coverage"`: dict mapping each field name to the fraction of records that have it (0.0-1.0)

In [ ]:
def audit_dataset(records: list[dict], schema: dict) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---
schema = {
    "id": {"type": str, "required": True},
    "caption": {"type": str, "required": True, "min_length": 5},
    "width": {"type": int, "required": True, "min": 256, "max": 4096},
    "height": {"type": int, "required": True, "min": 256, "max": 4096},
    "quality_score": {"type": float, "required": True, "min": 0.0, "max": 1.0},
    "source": {"type": str, "required": False},
}

records = [
    {"id": "img1", "caption": "a red car on road", "width": 512, "height": 512, "quality_score": 0.9, "source": "web"},  # valid
    {"id": "img2", "caption": "ok", "width": 1024, "height": 768, "quality_score": 0.7},  # caption too short
    {"id": "img3", "caption": "a sunset over ocean", "width": 100, "height": 512, "quality_score": 0.8},  # width too small
    {"id": "img4", "caption": "a dog playing fetch", "width": 512, "height": 512, "quality_score": 1.5},  # quality out of range
    {"id": "img1", "caption": "duplicate id test", "width": 256, "height": 256, "quality_score": 0.6},  # duplicate id
    {"id": "img5", "caption": "valid image here", "width": 512, "height": 512, "quality_score": 0.5, "source": "synthetic"},  # valid
    {"id": "img6", "width": 512, "height": 512, "quality_score": 0.8},  # missing caption
    {"id": "img7", "caption": "a nice photo", "width": "not_int", "height": 512, "quality_score": 0.8},  # wrong type
]

report = audit_dataset(records, schema)

assert report["total_records"] == 8
assert report["valid_records"] == 2, f"Expected 2 valid, got {report['valid_records']}"  # img1 (first) and img5

# Check specific issues
issues_by_idx = {}
for issue in report["issues"]:
    issues_by_idx.setdefault(issue["record_index"], []).append(issue)

assert 1 in issues_by_idx  # img2: caption too short
assert any(i["issue"] == "too_short" for i in issues_by_idx[1])
assert 2 in issues_by_idx  # img3: width too small
assert any(i["issue"] == "out_of_range" and i["field"] == "width" for i in issues_by_idx[2])
assert 6 in issues_by_idx  # img6: missing caption
assert any(i["issue"] == "missing" for i in issues_by_idx[6])
assert 7 in issues_by_idx  # img7: wrong type for width
assert any(i["issue"] == "wrong_type" for i in issues_by_idx[7])

# Duplicate IDs
assert "img1" in report["duplicate_ids"]

# Field coverage
assert report["field_coverage"]["id"] == 1.0  # all have id
assert report["field_coverage"]["source"] == 2/8  # only 2 have source

# Empty input
empty_report = audit_dataset([], schema)
assert empty_report["total_records"] == 0
assert empty_report["valid_records"] == 0

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: PyTorch Dataset with Transforms and Collation

**Likelihood: High** — Core PyTorch fluency test. Datasets role = you'll write these constantly.

### Scenario

Build a PyTorch Dataset for image-caption pairs with a custom collate function.

### Requirements

**`ImageCaptionDataset(Dataset)`:**
- `__init__(self, records: list[dict], transform=None)` — records have `"image"` (HxWx3 numpy uint8), `"caption"` (str)
- `__getitem__` returns `{"image": tensor, "tokens": tensor}` where:
  - image: float32, (3,H,W), values in [0,1] (divide by 255). Apply transform if provided.
  - tokens: int64 1D tensor from `tokenize(caption)` (provided below)
- `__len__` returns count

**`padded_collate(batch: list[dict]) -> dict`:**
- `"image"`: stack into (B,3,H,W) — assume same spatial size
- `"tokens"`: pad to longest sequence with 0, return (B, max_len)
- `"mask"`: attention mask (B, max_len) — 1 for real tokens, 0 for padding
- `"lengths"`: original sequence lengths as (B,) int64 tensor

```python
def tokenize(text: str) -> list[int]:
    return [hash(w) % 10000 + 1 for w in text.lower().split()]
```

In [ ]:
def tokenize(text: str) -> list[int]:
    return [hash(w) % 10000 + 1 for w in text.lower().split()]


class ImageCaptionDataset(Dataset):
    # YOUR CODE HERE
    pass


def padded_collate(batch: list[dict]) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 4 ---
records = [
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "a red car"},
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "a beautiful sunset over the ocean"},
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "dog"},
]

ds = ImageCaptionDataset(records)
assert len(ds) == 3

item = ds[0]
assert item["image"].shape == (3, 32, 32), f"Expected (3,32,32), got {item['image'].shape}"
assert item["image"].dtype == torch.float32
assert item["image"].min() >= 0.0 and item["image"].max() <= 1.0
assert item["tokens"].dtype == torch.int64
assert item["tokens"].ndim == 1
assert len(item["tokens"]) == 3  # "a red car" → 3 tokens

# With transform
def dummy_transform(img):
    return img * 2  # just scale

ds_t = ImageCaptionDataset(records, transform=dummy_transform)
item_t = ds_t[0]
assert item_t["image"].max() <= 2.0  # scaled by transform

# Collation
batch = [ds[i] for i in range(3)]
collated = padded_collate(batch)
assert collated["image"].shape == (3, 3, 32, 32)
assert collated["tokens"].shape[0] == 3
max_len = collated["tokens"].shape[1]
assert max_len == 6  # "a beautiful sunset over the ocean" → 6 tokens
assert collated["mask"].shape == (3, max_len)
assert collated["lengths"].tolist() == [3, 6, 1]

# Padding correctness
assert collated["mask"][2].sum().item() == 1  # "dog" → 1 token
assert (collated["tokens"][2, 1:] == 0).all()  # padded with 0
assert collated["mask"][1].sum().item() == 6  # longest, no padding

# DataLoader integration
loader = DataLoader(ds, batch_size=2, collate_fn=padded_collate)
batch_dl = next(iter(loader))
assert batch_dl["image"].shape[0] == 2

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Weighted Data Sampler

**Likelihood: High** — Job description: "how data composition affects model performance." Data mixing is central to the role.

### Scenario

You have training data from multiple sources (e.g., web images, synthetic, licensed). You need to sample batches with controlled proportions from each source.

### Requirements

**`DataMixer`:**
- `__init__(self, sources: dict[str, int])` — source_name → number of available items
- `set_ratios(self, ratios: dict[str, float])` — target sampling ratios (must sum to 1.0 ± 0.01, raise ValueError otherwise)
- `sample_batch(self, batch_size: int) -> dict[str, int]` — return how many items to draw from each source
  - Multiply ratio * batch_size, round using largest remainder method so the total equals exactly batch_size
  - If a source would need more than its available items, cap it and redistribute the excess proportionally to other sources
- `sample_epoch(self, epoch_size: int) -> list[dict[str, int]]` — generate batch allocations for an entire epoch
  - Track how many items have been "used" from each source
  - When a source is exhausted, redistribute its share proportionally to remaining sources
  - Return list of batch allocations (each is a dict like sample_batch returns)
  - Use batch_size=32 for each batch

In [ ]:
class DataMixer:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---
mixer = DataMixer({"web": 10000, "synthetic": 5000, "licensed": 1000})

# Set ratios
mixer.set_ratios({"web": 0.5, "synthetic": 0.3, "licensed": 0.2})

# Invalid ratios
try:
    mixer.set_ratios({"web": 0.5, "synthetic": 0.3, "licensed": 0.3})  # sums to 1.1
    assert False, "Should have raised ValueError"
except ValueError:
    pass

# Basic batch
batch = mixer.sample_batch(32)
assert sum(batch.values()) == 32, f"Batch should sum to 32, got {sum(batch.values())}"
assert batch["web"] == 16  # 0.5 * 32
assert batch["synthetic"] == 10  # round(0.3 * 32) = 9.6
assert batch["licensed"] == 6  # remainder

# Odd batch size — tests rounding
batch_odd = mixer.sample_batch(10)
assert sum(batch_odd.values()) == 10
assert batch_odd["web"] == 5  # 0.5 * 10
# 0.3 * 10 = 3.0, 0.2 * 10 = 2.0
assert batch_odd["synthetic"] == 3
assert batch_odd["licensed"] == 2

# Source exhaustion in sample_batch
tiny_mixer = DataMixer({"a": 3, "b": 10000})
tiny_mixer.set_ratios({"a": 0.8, "b": 0.2})
tiny_batch = tiny_mixer.sample_batch(10)
assert tiny_batch["a"] <= 3, f"Can't sample more than available: got {tiny_batch['a']}"
assert sum(tiny_batch.values()) == 10, "Should redistribute excess to other sources"

# Epoch sampling
small_mixer = DataMixer({"web": 50, "synthetic": 20})
small_mixer.set_ratios({"web": 0.6, "synthetic": 0.4})
epoch_batches = small_mixer.sample_epoch(epoch_size=64)
total_sampled = {"web": 0, "synthetic": 0}
for b in epoch_batches:
    for k, v in b.items():
        total_sampled[k] += v
assert sum(total_sampled.values()) == 64, f"Total epoch samples should be 64, got {sum(total_sampled.values())}"
# synthetic only has 20, so it should be exhausted and web should get more
assert total_sampled["synthetic"] <= 20

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Precision, Recall, F1 from Scratch

**Likelihood: High** — Job says "evaluations and benchmarks." Metrics are fundamental.

### Formulas (provided — you don't need to memorize these)

For each class `c`:
- **True Positives (TP):** items correctly predicted as class c
- **False Positives (FP):** items incorrectly predicted as class c (predicted c but actually another class)
- **False Negatives (FN):** items of class c that were predicted as another class

```
Precision(c) = TP / (TP + FP)    — of everything you predicted as c, how many were right?
Recall(c)    = TP / (TP + FN)     — of everything that IS c, how many did you find?
F1(c)        = 2 * P * R / (P + R) — harmonic mean (0 if P + R = 0)

Macro F1     = mean of per-class F1
Weighted F1  = sum(F1(c) * support(c)) / total_samples
              where support(c) = number of true instances of class c
```

### Requirements

**`compute_metrics(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int) -> dict`:**
- Both are 1D int64 tensors
- Return dict with: `"per_class_precision"`, `"per_class_recall"`, `"per_class_f1"` (each a list of floats), `"macro_f1"` (float), `"weighted_f1"` (float)
- Handle edge cases: if a class has no predictions, precision = 0. If no ground truth, recall = 0.

In [ ]:
def compute_metrics(
    predictions: torch.Tensor, targets: torch.Tensor, num_classes: int
) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 6 ---

# Binary case
preds = torch.tensor([0, 0, 1, 1, 1, 0])
targs = torch.tensor([0, 1, 1, 1, 0, 0])
result = compute_metrics(preds, targs, num_classes=2)

# Class 0: TP=2, FP=1, FN=1 → P=2/3, R=2/3, F1=2/3
# Class 1: TP=2, FP=1, FN=1 → P=2/3, R=2/3, F1=2/3
assert abs(result["per_class_precision"][0] - 2/3) < 1e-5
assert abs(result["per_class_recall"][0] - 2/3) < 1e-5
assert abs(result["per_class_f1"][0] - 2/3) < 1e-5
assert abs(result["macro_f1"] - 2/3) < 1e-5

# Multi-class
preds_mc = torch.tensor([0, 1, 2, 0, 1, 2, 0, 2])
targs_mc = torch.tensor([0, 1, 2, 0, 2, 1, 1, 2])
result_mc = compute_metrics(preds_mc, targs_mc, num_classes=3)
assert len(result_mc["per_class_f1"]) == 3
assert 0 <= result_mc["macro_f1"] <= 1
assert 0 <= result_mc["weighted_f1"] <= 1

# Edge: class with no predictions
preds_edge = torch.tensor([0, 0, 0])
targs_edge = torch.tensor([0, 1, 0])
result_edge = compute_metrics(preds_edge, targs_edge, num_classes=2)
assert result_edge["per_class_precision"][1] == 0.0  # no predictions for class 1

# Perfect predictions
perfect = compute_metrics(torch.tensor([0, 1, 2]), torch.tensor([0, 1, 2]), num_classes=3)
assert abs(perfect["macro_f1"] - 1.0) < 1e-5

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Scaled Dot-Product Attention

**Likelihood: Moderate-High** — Common ML coding question. Shows you understand the core transformer building block.

### Formula (provided)

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

For **multi-head attention** with h heads and model dimension d:
1. Project input X through W_q, W_k, W_v (each d → d)
2. Split into h heads: reshape (B, T, d) → (B, h, T, d/h)
3. Apply attention formula per head
4. Concatenate heads: (B, h, T, d/h) → (B, T, d)
5. Final projection through W_o (d → d)

**Causal mask:** Set positions where query_pos < key_pos to -inf before softmax.

### Requirements

**`MultiHeadAttention(nn.Module)`:**
- `__init__(self, d_model: int, num_heads: int)` — assert d_model % num_heads == 0
- Use a single linear (no bias) for Q, K, V projection (d_model → 3 * d_model), and one for output (d_model → d_model)
- `forward(self, x, causal=False)` — x shape (B,T,D), output (B,T,D)

Do NOT use `nn.MultiheadAttention` or `F.scaled_dot_product_attention`.

In [ ]:
class MultiHeadAttention(nn.Module):
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 7 ---
torch.manual_seed(42)

mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)

# Basic forward
out = mha(x)
assert out.shape == (2, 10, 64), f"Expected (2,10,64), got {out.shape}"

# Causal forward
out_causal = mha(x, causal=True)
assert out_causal.shape == (2, 10, 64)

# Causal property: output at position t should not depend on positions > t
x_test = torch.randn(1, 5, 64)
out1 = mha(x_test, causal=True)
x_modified = x_test.clone()
x_modified[0, 3:, :] = torch.randn(2, 64)  # Change positions 3,4
out2 = mha(x_modified, causal=True)
# Positions 0,1,2 should be identical
assert torch.allclose(out1[0, :3], out2[0, :3], atol=1e-5), "Causal mask broken"
# Position 3 should differ
assert not torch.allclose(out1[0, 3], out2[0, 3], atol=1e-3)

# Parameter count: W_qkv (d * 3d) + W_o (d * d), no bias
total_params = sum(p.numel() for p in mha.parameters())
assert total_params == 64 * 64 * 3 + 64 * 64, f"Unexpected param count: {total_params}"

# Gradient flows
loss = out.sum()
loss.backward()
for name, p in mha.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"

# Invalid config
try:
    MultiHeadAttention(d_model=63, num_heads=8)
    assert False, "Should have raised error"
except (AssertionError, ValueError):
    pass

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Simple Training Pipeline

**Likelihood: Moderate** — Tests PyTorch fluency. Shows you can wire up the standard train/eval loop.

### Requirements

**`Trainer`:**
- `__init__(self, model, optimizer, train_loader, val_loader)`
- `train_epoch(self) -> float` — run one training epoch, return mean loss
  - For each batch: forward → loss → backward → optimizer step → zero_grad
  - The model's forward takes a batch (tuple) and returns `(loss, logits)`
- `evaluate(self) -> float` — run validation, return mean loss (no gradients!)
- `fit(self, num_epochs: int, patience: int) -> dict` — train with early stopping
  - After each epoch, evaluate on validation set
  - If val loss hasn't improved for `patience` epochs, stop early
  - Track best val loss and the epoch it occurred at
  - Return `{"train_losses": list[float], "val_losses": list[float], "best_epoch": int, "stopped_early": bool}`

In [ ]:
class Trainer:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 8 ---

# Simple regression model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
    def forward(self, batch):
        x, y = batch
        pred = self.linear(x).squeeze(-1)
        loss = F.mse_loss(pred, y)
        return loss, pred

torch.manual_seed(42)
model = SimpleModel()

# Create data with learnable pattern
X = torch.randn(200, 10)
true_w = torch.randn(10)
y = X @ true_w + 0.1 * torch.randn(200)

train_ds = torch.utils.data.TensorDataset(X[:160], y[:160])
val_ds = torch.utils.data.TensorDataset(X[160:], y[160:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
trainer = Trainer(model, optimizer, train_loader, val_loader)

# Single epoch
train_loss = trainer.train_epoch()
assert isinstance(train_loss, float)
assert train_loss > 0

val_loss = trainer.evaluate()
assert isinstance(val_loss, float)
assert val_loss > 0

# Model should be in eval mode after evaluate
# but back in train mode shouldn't be required

# Full training with early stopping
torch.manual_seed(42)
model2 = SimpleModel()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)
trainer2 = Trainer(model2, optimizer2, train_loader, val_loader)

result = trainer2.fit(num_epochs=50, patience=5)
assert "train_losses" in result
assert "val_losses" in result
assert "best_epoch" in result
assert "stopped_early" in result
assert len(result["train_losses"]) == len(result["val_losses"])

# Loss should decrease
assert result["train_losses"][-1] < result["train_losses"][0], "Training loss should decrease"

# If stopped early, should have run fewer than 50 epochs
if result["stopped_early"]:
    assert len(result["train_losses"]) < 50
    # Should have stopped patience epochs after best
    assert len(result["train_losses"]) <= result["best_epoch"] + 5 + 1

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

### Warmups (Part A)

| Result | Assessment |
|--------|------------|
| < 5 min, first try | **Automatic** — this pattern is locked in |
| 5-10 min | **Solid** — you'd be fine on a CoderPad |
| 10-15 min or needed to debug | **Shaky** — drill this one again tomorrow |
| Couldn't solve | **Gap** — review the concept, then redo |

### Full Problems (Part B)

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** — you'd ace this question |
| Solved in 25-30 min, minor debugging | **Pass** — exam-ready |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Drill this** — break it into pieces, understand each part |

### Daily Practice

1. **Warmup round (20 min):** Do 2-3 warmups you haven't nailed yet
2. **Main problem (30 min):** Pick 1 full problem, set timer, go
3. **Review (10 min):** If you failed, identify the specific sticking point (tensor shapes? API? logic?) and note it for tomorrow

**Priority order:** Problems 1-5 > Problems 6-8 > Warmups you've already nailed